In [1]:
import os
import sys
from tqdm import tqdm
import pickle
import numpy as np

import torch
from scipy.stats import zscore
from torchvision import transforms
sub_list = ['sub-01', 'sub-02', 'sub-04']
sessions = ['01', '02', '03', 'study', 'test', 'snap']

In [3]:
dic = {}

data_folder = '/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/'

folder_path = os.path.join(data_folder, 'afni')
for sub in sub_list:
    with open(f'{folder_path}/{sub}_roi_vox_all_sessions.pkl', 'rb') as file:
        dic[sub] = pickle.load(file)

In [4]:
# Masking data
for sub in sub_list:
    union_mask = dic[sub]['union_mask']
    print('NSD mask size:', union_mask.shape)
    print('union mask size:', sum(union_mask))
    for ses in sessions:
        dic[sub][ses]['roi'] = dic[sub][ses]['roi'][:, union_mask]
        # z-score each session
        dic[sub][ses]['roi'] = np.nan_to_num(zscore(dic[sub][ses]['roi'], axis=0))
        s = dic[sub][ses]['roi'].shape
        print(f'{ses}: {s}')

NSD mask size: (16946,)
union mask size: 5796
01: (693, 5796)
02: (693, 5796)
03: (693, 5796)
study: (216, 5796)
test: (216, 5796)
snap: (432, 5796)
NSD mask size: (17175,)
union mask size: 3643
01: (693, 3643)
02: (693, 3643)
03: (693, 3643)
study: (216, 3643)
test: (216, 3643)
snap: (432, 3643)
NSD mask size: (20397,)
union mask size: 5887
01: (693, 5887)
02: (693, 5887)
03: (693, 5887)
study: (216, 5887)
test: (216, 5887)
snap: (432, 5887)


In [5]:
# 455 * 3 + 26 (13 pairs; 3 repeats) + 80 (2 repeats per session_
unique_images = set(dic[sub]['01']['trial'] + dic[sub]['02']['trial'] + dic[sub]['03']['trial'])

In [6]:
import imageio.v2 as imageio
resize_transform = transforms.Resize((224, 224))

images = None

for img in tqdm(unique_images):
    
    root_dir = os.path.join(data_folder, 'stimuli')
    if 'unchosen' in img:
        image_file = f'{root_dir}/unchosen_nsd_1000_images/{img}.png'
    elif 'special' in img and 'notspecial' not in img:
        image_file = f'{root_dir}/special515/{img}.jpg'
    elif 'notspecial' in img:
        image_file = f'{root_dir}/shared1000_notspecial/{img}.png'
    elif 'pair_' and '_w_' in img:
        image_file = f'{root_dir}/MST_pairs/{img}.jpg'
    else:
        print(img)
        
    if image_file and not os.path.exists(image_file):
        print('Cannot find the image at this path',image_file)
        break
        
    im = imageio.imread(image_file)
    im = torch.Tensor(im / 255).permute(2,0,1)
    im = resize_transform(im.unsqueeze(0))
    
    if images is None:
        images = im
    else:
        images = torch.vstack((images, im))
        
print("images", images.shape)

  0%|          | 1/1471 [00:00<03:54,  6.28it/s]/home/wg7536/.conda/envs/rt_mindEye2/lib/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
100%|██████████| 1471/1471 [02:27<00:00,  9.99it/s]

images torch.Size([1471, 3, 224, 224])


In [7]:
def find_repeated_strings(string_list):
    """
    Finds all repeated strings in a list and returns a dictionary 
    with the string as the key and a list of its indices as the value.
    Uses a set to track seen items efficiently.
    """
    # Set to quickly track which items have appeared once already
    seen_once = set()
    # Dictionary to store only the indices of items that repeat
    repeated_strings_dict = {}

    for index, string_val in enumerate(string_list):
        if string_val in repeated_strings_dict:
            # If already in the 'repeated_strings_dict', just append the new index
            repeated_strings_dict[string_val].append(index)
        elif string_val in seen_once:
            # First time seeing a repeat: move from 'seen_once' to 'repeated_strings_dict'
            repeated_strings_dict[string_val] = [string_list.index(string_val), index]
        else:
            # First time seeing the item overall
            seen_once.add(string_val)
            
    return repeated_strings_dict

In [8]:
def average_repeats(vox, mindeye_trial, unique_images):
    
    repeated_trial = find_repeated_strings(mindeye_trial)
    
    sorted_vox = np.zeros((len(unique_images), vox.shape[1]))
    
    # Average repeated MST images
    for i, img in enumerate(unique_images):

        if img in repeated_trial.keys(): # deal with repeated images
            # average all repeats across sessions
            curr_trial_vox = np.mean(vox[repeated_trial[img]], axis=0)
            
        elif img in mindeye_trial: # deal with once images
            idx = mindeye_trial.index(img)
            curr_trial_vox = vox[idx, :]
            
        else: # error handeling
            print(f"{img} is not in the list")
            break
        
        sorted_vox[i, :] = curr_trial_vox
    
    return sorted_vox

In [9]:
# Stacking multi-session data:
vox_data = {}
for sub in sub_list:
    mindeye_vox = np.vstack((dic[sub]['01']['roi'],dic[sub]['02']['roi'],dic[sub]['03']['roi']))
    mindeye_trial = dic[sub]['01']['trial']+dic[sub]['02']['trial']+dic[sub]['03']['trial']
    mindeye_vox = average_repeats(mindeye_vox, mindeye_trial, unique_images)
    vox_data[sub] = mindeye_vox

In [10]:
images = torch.Tensor(images)
vox = torch.Tensor(vox_data['sub-01'])
assert len(images) == len(vox)

## Finished loading data. Setting up GPU

In [11]:
### Multi-GPU config ###
from accelerate import Accelerator, DeepSpeedPlugin

local_rank = os.getenv('RANK')
if local_rank is None: 
    local_rank = 0
else:
    local_rank = int(local_rank)
print("LOCAL RANK ", local_rank)  

data_type = torch.float32 # change depending on your mixed_precision

accelerator = Accelerator(split_batches=False)
batch_size = 8 

LOCAL RANK  0


In [12]:
print("PID of this process =",os.getpid())
device = accelerator.device
print("device:",device)
world_size = accelerator.state.num_processes
distributed = not accelerator.state.distributed_type == 'NO'
num_devices = torch.cuda.device_count()
global_batch_size = batch_size * num_devices
print("global_batch_size", global_batch_size)
if num_devices==0 or not distributed: num_devices = 1
num_workers = num_devices
print(accelerator.state)

# set data_type to match your mixed precision (automatically set based on deepspeed config)
if accelerator.mixed_precision == "bf16":
    data_type = torch.bfloat16
elif accelerator.mixed_precision == "fp16":
    data_type = torch.float16
else:
    data_type = torch.float32

print("distributed =",distributed, "num_devices =", num_devices, "local rank =", local_rank, "world size =", world_size, "data_type =", data_type)
print = accelerator.print # only print if local_rank=0

PID of this process = 681070
device: cuda
global_batch_size 8
Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: no

distributed = False num_devices = 1 local rank = 0 world size = 1 data_type = torch.float32


In [13]:
## USING OpenCLIP ViT-bigG ###
sys.path.append('generative_models/')
import sgm
from generative_models.sgm.modules.encoders.modules import FrozenOpenCLIPImageEmbedder
# from generative_models.sgm.models.diffusion import DiffusionEngine
# from omegaconf import OmegaConf

In [14]:
try:
    print(clip_img_embedder)
except:
    clip_img_embedder = FrozenOpenCLIPImageEmbedder(
        arch="ViT-bigG-14",
        version="laion2b_s39b_b160k",
        output_tokens=True,
        only_tokens=True,
    )
    clip_img_embedder.to(device)
clip_seq_dim = 256
clip_emb_dim = 1664

In [15]:
num_voxels_list=[vox[0].shape[-1]]
n_blocks=4
hidden_dim=1024
use_prior=True
clip_scale=1.

In [16]:
import utils
from models import PriorNetwork, BrainDiffusionPrior

In [17]:
model = utils.prepare_model_and_training(
    num_voxels_list=num_voxels_list,
    n_blocks=n_blocks,
    hidden_dim=hidden_dim,
    clip_emb_dim=clip_emb_dim,
    clip_seq_dim=clip_seq_dim,
    use_prior=use_prior,
    clip_scale=clip_scale
)

MindEyeModule()
param counts:
5,936,128 total
5,936,128 trainable
param counts:
5,936,128 total
5,936,128 trainable
param counts:
453,360,280 total
453,360,280 trainable
param counts:
459,296,408 total
459,296,408 trainable
param counts:
259,865,216 total
259,865,200 trainable
param counts:
719,161,624 total
719,161,608 trainable
